# Example of Information Gain

- This notebook illustrates the computation of information gain.
- Information gain is a fundamental criterion in the construction of decision trees, especially in entropy-based algorithms such as ID3.
- The notebook uses the restaurant example described in Chapter 19 of the AIMA book (Artificial Intelligence: A Modern Approach, 4th edition).
- This code does not build the full tree and is designed for clarity rather than optimization. Its main purpose is to illustrate the process of computing information gain..

# Variables

Variables:
1. Alternate: whether there is a suitable alternative restaurant nearby.
2. Bar: whether the restaurant has a comfortable bar area to wait in.
3. Fri/Sat: true on Fridays and Saturdays.
4. Hungry: whether we are hungry right now.
5. Patrons: how many people are in the restaurant (values are None, Some, and Full).
6. Price: the restaurant’s price range ($, $$, $$$).
7. Raining: whether it is raining outside.
8. Reservation: whether we made a reservation.
9. Type: the kind of restaurant (French, Italian, Thai, or burger).
10. WaitEstimate: host’s wait estimate: 0–10, 10–30, 30–60, or >60 minutes.



# Data

In [ ]:
import pandas as pd
import math

headers = [
    "Example", "Alt", "Bar", "Fri", "Hun", "Pat",
    "Price", "Rain", "Res", "Type", "Est", "WillWait"
]

data = [
    ["x1",  "Yes", "No",  "No",  "Yes", "Some", "$$$", "No",  "Yes", "French",  "0-10",  "Yes"],
    ["x2",  "Yes", "No",  "No",  "Yes", "Full", "$",   "No",  "No",  "Thai",    "30-60", "No"],
    ["x3",  "No",  "Yes", "No",  "No",  "Some", "$",   "No",  "No",  "Burger",  "0-10",  "Yes"],
    ["x4",  "Yes", "No",  "Yes", "Yes", "Full", "$",   "Yes", "No",  "Thai",    "10-30", "Yes"],
    ["x5",  "Yes", "No",  "Yes", "No",  "Full", "$$$", "No",  "Yes", "French",  ">60",   "No"],
    ["x6",  "No",  "Yes", "No",  "Yes", "Some", "$$",  "Yes", "Yes", "Italian", "0-10",  "Yes"],
    ["x7",  "No",  "Yes", "No",  "No",  "None", "$",   "Yes", "No",  "Burger",  "0-10",  "No"],
    ["x8",  "No",  "No",  "No",  "Yes", "Some", "$$",  "Yes", "Yes", "Thai",    "0-10",  "Yes"],
    ["x9",  "No",  "Yes", "Yes", "No",  "Full", "$",   "Yes", "No",  "Burger",  ">60",   "No"],
    ["x10", "Yes", "Yes", "Yes", "Yes", "Full", "$$$", "No",  "Yes", "Italian", "10-30", "No"],
    ["x11", "No",  "No",  "No",  "No",  "None", "$",   "No",  "No",  "Thai",    "0-10",  "No"],
    ["x12", "Yes", "Yes", "Yes", "Yes", "Full", "$",   "No",  "No",  "Burger",  "30-60", "Yes"],
]

df = pd.DataFrame(data, columns=headers)

print(df)

# Auxiliar functions and classes

In [ ]:
def binary_entropy(p):
    if p == 0 or p == 1:
        H = 0
    else:
        H = -p * math.log2(p) - (1 - p) * math.log2(1 - p)
    return H

class Split():
    def __init__(self, name, p, n, df_where_value):
        
        pk = (df_where_value["WillWait"] == "Yes").sum()
        nk = (df_where_value["WillWait"] == "No").sum()

        self.name = name 
        self.pk = pk 
        self.nk = nk 
        self.weight = (pk + nk) / ( p + n)  
        self.pos_prob = pk / (pk + nk)
        self.entropy = binary_entropy(pk / (pk + nk))

    def show(self):
        msg = (
            "    {:<10} "
            "pk,nk: {:>2},{:<2}   "
            "pos_prob: {:>5.2f}   "
            "weight: {:>5.2f}   "
            "Entropy: {:>5.2f}"
        ).format(
            self.name,
            self.pk,
            self.nk,
            self.pos_prob,
            self.weight,
            self.entropy
        )

        print(msg)


# Compute the initial entropy

In [ ]:
# Initial entropy of the initial distribution of clases
# Remember, entropy is 
# H = -p(yes)*log_2[p(yes)] - p(no)*log_2(p(no)). Alternatively, We can uso use the binary_entropy function

# parent distribution
target_variable = "WillWait"

p = (df[target_variable] == "Yes").sum()   # positive samples
n = (df[target_variable] == "No").sum()    # negative samples

H_initial = binary_entropy(p / len(df))     # Initial entropy
print(f"Initial entropy: {H_initial} bits")

# Compute information gain

In [ ]:
columns = df.columns.tolist()
columns.pop(0)
columns.pop(-1)

attributes = columns

information_gain = {}

for attribute in attributes:
    print(attribute)
    cardinality = df[attribute].unique() 
    atribute_splits = []

    for value in df[attribute].unique():
        # filter examples where Pat = value
        df_where_value = df[df[attribute] == value]  

        split = Split(value, p, n, df_where_value)
        #atribute_splits[attribute] = split
        atribute_splits.append(split)

    remaining_entropy = 0
    for split in atribute_splits:
        split.show()
        
        # compute remaining entropy
        remaining_entropy += split.weight * binary_entropy(split.pos_prob)

    information_gain[attribute] = H_initial - remaining_entropy


# Print the information gain

In [ ]:
for k,v in information_gain.items():
    print('{}: \t {:.2f}'.format(k,v))